# LineFormer probe — batch chart→data on Colab GPU

**Goal:** judge published-LineFormer quality on OUR catalysis figures before
investing in a standalone rewrite. Zero cost: free T4 GPU.

**You need:** `figures/lineformer_probe.zip` from the repo machine (30 images:
10 figures as full + single-panel crops).

Runtime → Change runtime type → **T4 GPU**, then run cells top to bottom.

**VS Code Colab-kernel users:** Select Kernel → Colab works; when CREATING the
server pick a **T4 GPU** shape (the default is CPU-only). Cells 6/8 fall back
to a Google-Drive transfer automatically when the web-frontend file widgets
are unavailable.

In [ ]:
# 1. GPU check
import torch
print('torch', torch.__version__, '| cuda', torch.version.cuda, '| GPU:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU — set Runtime > Change runtime type > T4 GPU'

In [ ]:
# 2. Clone the pipeline repo (LineFormer + ChartDete wrapper)
%cd /content
!rm -rf extract-line-chart-data
!git clone -q https://github.com/tdsone/extract-line-chart-data.git
%cd extract-line-chart-data
!ls

In [ ]:
# 3. Install — mmcv/mmdet pinned to THIS Colab's torch up front.
#    This is the fragile step; expect ~5-10 min. Errors about 'mmcv version'
#    at import time are what cell 4 is for.
import torch
cu = 'cu' + torch.version.cuda.replace('.', '')
tv = torch.__version__.split('+')[0]
print(f'pinning mmcv for torch{tv}/{cu}')
!pip install -q openmim mmengine
!mim install -q "mmcv>=2.0.0,<2.2.0"
!pip install -q -e ".[local]"
!bash setup_local_env.sh || echo '--- setup_local_env.sh reported errors: read them before continuing ---'

### 4. If the install fights the current torch
MMDetection-era code may need an older torch. Nuclear option (slow, ~5 min):
```python
!pip install -q torch==2.1.2 torchvision==0.16.2 --index-url https://download.pytorch.org/whl/cu121
!pip install -q mmcv==2.1.0 -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.1/index.html
```
then **Runtime → Restart session** and re-run from cell 2 (skip the pin in cell 3).

In [ ]:
# 5. Pre-download the model weights. The wrapper lazy-downloads them from
#    HuggingFace (tdsone/lineformer, tdsone/chartdete) into
#    ~/.cache/plextract/ on FIRST inference — fetching them here makes the
#    download visible and fails fast on network problems.
from huggingface_hub import snapshot_download
from pathlib import Path
for repo, sub in [("tdsone/lineformer", "lineformer"),
                  ("tdsone/chartdete", "chartdete")]:
    d = Path.home() / ".cache" / "plextract" / sub
    if not d.exists():
        print(f"downloading {repo} ...")
        snapshot_download(repo, local_dir=str(d))
    print(sub, "->", sorted(p.name for p in d.iterdir()))
import glob, os
assert glob.glob(os.path.expanduser("~/.cache/plextract/*/*.pth")), "weights missing"
print("checkpoints ready")

In [ ]:
# 6. Get the probe zip onto the VM.
#    DRIVE_LINK is pre-filled (a shared Drive FOLDER containing
#    lineformer_probe.zip); set it to '' to use the web-UI file picker instead.
import os, glob, zipfile, subprocess
os.makedirs('input', exist_ok=True)

DRIVE_LINK = 'https://drive.google.com/drive/folders/1YwfgY43hBMLESwZZrYpKiiKFXYPC1CCL?usp=sharing'

zname = None
if DRIVE_LINK:
    subprocess.run(['pip', 'install', '-q', 'gdown'], check=True)
    import gdown
    if '/folders/' in DRIVE_LINK:
        gdown.download_folder(url=DRIVE_LINK, output='drive_in', quiet=False)
        hits = glob.glob('drive_in/**/*probe*.zip', recursive=True) \
               or glob.glob('drive_in/**/*.zip', recursive=True)
        assert hits, 'no zip found in the shared folder'
        zname = hits[0]
    else:
        zname = gdown.download(url=DRIVE_LINK, output='lineformer_probe.zip',
                               fuzzy=True, quiet=False)
else:
    from google.colab import files          # web frontend only
    up = files.upload()
    zname = next(iter(up))

with zipfile.ZipFile(zname) as z:
    z.extractall('input')
pngs = [f for f in os.listdir('input') if f.endswith('.png')]
print(f'input/: {len(pngs)} images')

In [ ]:
# 7. Batch extract (single-panel crops are the intended input; the *__full*
#    images are there to see how it copes with composites)
from plextract import extract
extract(input_dir='input', output_dir='output', backend='local')
!find output -name '*.json' | head -20

In [ ]:
# 8. Zip the results and get them back to your machine.
import shutil
shutil.make_archive('lineformer_probe_results', 'zip', 'output')
try:
    from google.colab import files
    files.download('lineformer_probe_results.zip')
except Exception as e:
    print('files.download() unavailable (VS Code kernel?):', e)
    # FALLBACK A: mount Drive and copy (auth prompt should appear once):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        shutil.copy('lineformer_probe_results.zip',
                    '/content/drive/MyDrive/lineformer_probe_results.zip')
        print('-> saved to Drive: MyDrive/lineformer_probe_results.zip '
              '(download it from drive.google.com)')
    except Exception as e2:
        print('Drive mount also failed:', e2)
        print('FALLBACK B: right-click lineformer_probe_results.zip in the '
              'VS Code/Colab file browser and download it manually.')
print('then drop the zip on the repo machine as figures/lineformer_probe_results.zip')

## What happens next (on the repo machine)
Claude fuses the LineFormer pixel traces with the existing axis-calibration
layer and rebuilds the verification HTML — LineFormer line extraction
side-by-side with the originals and with the CV reader, so the quality call
is made by eye on our own figures.

**Judging criteria:** (1) right number of curves per panel — especially on the
GRAYSCALE marker figures (catcom_2009, jcat_2015.01) where the CV reader is
blind; (2) traces follow the printed curves through crossings; (3) composite
(*__full*) behaviour vs pre-split panels.